In [ ]:
# 一、准备文档
from langchain_text_splitters import MarkdownHeaderTextSplitter


def prepare_docs():
    # =============1.准备文档================
    with open('./评估.md', 'r', encoding='utf-8') as f:
        doc = f.read()

    # =============2.切分文档================
    # 切分依据，这里是按照三级标题
    headers_to_split_on = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]

    # 创建切分器
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on,
        strip_headers=False  # 在正文中是否去除标题，False，不去除
    )

    # 切分文档, 返回的是 list[Document]
    docs = markdown_splitter.split_text(doc)

    # 为了方便查看，给每个 Document 加一个id，从 1 开始递增，但要注意的是 LangChain 的文档 id 要求是 str
    for i, doc in enumerate(docs, start=1):
        doc.id = str(i)
    return docs

docs = prepare_docs()

In [ ]:
# 二、初始化向量模型
from langchain_ollama import OllamaEmbeddings
# 1.初始化向量模型
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)

In [ ]:
# 三、初始化Collection，或者叫VectorStore
from langchain_milvus import Milvus, BM25BuiltInFunction
from app.core.config import settings
collection_name = "LangChain_Milvus66"

#embedding_function ,builtin_function 因为这两个都设置了,所以检索时为混合检索,否则只传其一
vectorstore = Milvus(
    embedding_function = embeddings,        # 稠密向量模型
    collection_name=collection_name,        # collection名称
    builtin_function=BM25BuiltInFunction(   # 生成稀疏向量的函数
        analyzer_params={"type": "chinese"} # 指定中文分词
    ),
    vector_field=["dense", "sparse"],       # 向量字段，包括稠密和稀疏
    connection_args={
        "uri": settings.rag.milvus_url,     # milvus的uri路径
    },
    # index_params=[index_param],             # 可选，索引参数，必须与vector_field对应
    drop_old=True,                          # 是否删除旧的collection，避免重复创建
)

In [ ]:
# 四、添加数据
#传入list[Document]即可，LangChain会自动根据Document生成Schema、Index、Collection
vectorstore.add_documents(docs)

In [ ]:
# 五、基于权重的混合检索
query = "给教师的建议是谁写的"

# 带得分的相似度检索，返回值是list[tuple[Document, float]]
result = vectorstore.similarity_search_with_score(
    query,
    ranker_type="weighted",
    ranker_params={"weights": [0.4, 0.6]}
)

for doc, score in result:
    print(f"============score: {score}==============")
    print(doc.model_dump_json(indent=2))